
# D-CQHGT Disruption Cold-OD

This notebook performs the next two tasks in the FREIGHT-MNet workflow:

1. **Feature pruning and disruption-group ablation.**
   It reads the OD-year disruption features created by `Build_Disruption_Features_ODYear.ipynb`, removes zero / constant / source-QA columns, and builds interpretable disruption groups such as weather-zone, weather-global, rail-global, border-global, and full variable disruption features.

2. **D-CQHGT disruption-gate training on Cold-OD.**
   It trains disruption-calibrated HGT models on two stable topology candidates:
   - an overall candidate: `hgt_truck_rail_plus_topology_features`
   - a sparse/risk candidate: `hgt_terminal_plus_truck_plus_rail`

The output target remains annual FAF OD truck travel-time quantiles:

```text
truck_q25, truck_q50, truck_q75
```

The main split is the existing **Cold-OD split**, where validation and test OD pairs are unseen during training.



## 1. Kernel preflight

This notebook requires the CUDA/PyG environment. If this cell reports that the kernel is not the expected environment, switch the Jupyter kernel to `Python (freight_mnet_cuda)` before running the rest of the notebook.

This preflight is deliberately placed before importing pandas, torch, or torch_geometric, because the base Anaconda environment has repeatedly shown NumPy ABI and missing PyG issues.


In [1]:

# ---------------------------------------------------------------------
# Kernel preflight: verify that Jupyter is using the FREIGHT-MNet CUDA/PyG environment.
# ---------------------------------------------------------------------

import os
import sys
import subprocess
from pathlib import Path

EXPECTED_PYTHON = Path(r"E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe")
current_python = Path(sys.executable)

print("Current Python executable:", current_python)
print("Expected FREIGHT-MNet CUDA/PyG Python:", EXPECTED_PYTHON)

if EXPECTED_PYTHON.exists() and current_python.resolve() != EXPECTED_PYTHON.resolve():
    print("\nWARNING: This notebook is not running in the expected CUDA/PyG environment.")
    print("Attempting to register the expected environment as a Jupyter kernel.")
    try:
        subprocess.run(
            [str(EXPECTED_PYTHON), "-m", "pip", "install", "ipykernel"],
            check=False,
            capture_output=True,
            text=True,
        )
        subprocess.run(
            [
                str(EXPECTED_PYTHON),
                "-m",
                "ipykernel",
                "install",
                "--user",
                "--name",
                "freight_mnet_cuda",
                "--display-name",
                "Python (freight_mnet_cuda)",
            ],
            check=False,
            capture_output=True,
            text=True,
        )
        print("Kernel registration attempted. Please switch to: Python (freight_mnet_cuda)")
    except Exception as exc:
        print("Kernel registration failed:", exc)

    raise RuntimeError(
        "Wrong Jupyter kernel. Switch to Python (freight_mnet_cuda), then restart and rerun this notebook."
    )
else:
    print("Kernel preflight passed.")


Current Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Expected FREIGHT-MNet CUDA/PyG Python: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Kernel preflight passed.



## 2. Imports and global environment settings

This cell imports the scientific stack and disables optional pandas acceleration backends that can cause issues in mixed NumPy environments. The notebook avoids `DataFrame.to_markdown()` so it does not require the optional `tabulate` package.


In [2]:

from __future__ import annotations

import json
import math
import os
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.nn import HGTConv
except Exception as exc:
    raise ImportError(
        "torch_geometric is required. Activate the FREIGHT-MNet CUDA/PyG environment."
    ) from exc

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False

print("Python:", os.sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


Python: 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
NumPy: 2.4.5
Pandas: 2.3.3
Torch: 2.12.0+cu126
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4050 Laptop GPU



## 3. Experiment configuration

The configuration keeps all paths explicit and reproducible. The default scope is East-Plus-Gulf, because this is the primary scope used throughout the experiments.

For a quick smoke test, set `seeds=(42,)`, reduce `max_epochs`, and reduce `enabled_disruption_groups`. For the full experiment, use the default seeds and groups.


In [3]:

# ---------------------------------------------------------------------
# Experiment configuration.
# ---------------------------------------------------------------------

@dataclass
class ExperimentConfig:
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")
    scope: str = "east_plus_gulf"
    run_name: str = "dcqhgt_disruption_cold_od_v1_notebook"

    # Reproducibility.
    seeds: Tuple[int, ...] = (7, 42, 123, 2026, 535)

    # Training hyperparameters.
    hidden_dim: int = 128
    num_layers: int = 2
    num_heads: int = 4
    dropout: float = 0.10
    lr: float = 1e-3
    weight_decay: float = 1e-4
    batch_size: int = 4096
    max_epochs: int = 300
    patience: int = 40
    grad_clip_norm: float = 5.0
    lambda_iqr: float = 0.10
    disruption_gate_alpha: float = 1.0

    # Sample weights.
    weight_column: str = "obs_weight_sum"
    weight_clip_min: float = 0.05
    weight_clip_max: float = 20.0

    # Disruption feature pruning.
    max_missing_rate: float = 0.05
    min_std: float = 1e-10
    min_nonzero_rate: float = 0.0

    # Model variant controls.
    enabled_topology_candidates: Tuple[str, ...] = (
        "hgt_truck_rail_plus_topology_features",
        "hgt_terminal_plus_truck_plus_rail",
    )
    enabled_disruption_groups: Tuple[str, ...] = (
        "no_disruption",
        "weather_zone_only",
        "weather_global_only",
        "weather_all",
        "rail_global_only",
        "border_global_only",
        "weather_zone_plus_rail_global",
        "weather_zone_plus_border_global",
        "full_disruption_variable_only",
    )

    # Checkpoint metrics.
    checkpoint_metrics: Tuple[str, ...] = ("best_val_pinball", "best_val_q75", "best_val_iqr")

    # Output and plotting.
    overwrite: bool = True
    save_models: bool = True
    make_plots: bool = True

cfg = ExperimentConfig()
print(cfg)


ExperimentConfig(data_root=WindowsPath('E:/NetworkOptimization/pythonProject1/Data'), scope='east_plus_gulf', run_name='dcqhgt_disruption_cold_od_v1_notebook', seeds=(7, 42, 123, 2026, 535), hidden_dim=128, num_layers=2, num_heads=4, dropout=0.1, lr=0.001, weight_decay=0.0001, batch_size=4096, max_epochs=300, patience=40, grad_clip_norm=5.0, lambda_iqr=0.1, disruption_gate_alpha=1.0, weight_column='obs_weight_sum', weight_clip_min=0.05, weight_clip_max=20.0, max_missing_rate=0.05, min_std=1e-10, min_nonzero_rate=0.0, enabled_topology_candidates=('hgt_truck_rail_plus_topology_features', 'hgt_terminal_plus_truck_plus_rail'), enabled_disruption_groups=('no_disruption', 'weather_zone_only', 'weather_global_only', 'weather_all', 'rail_global_only', 'border_global_only', 'weather_zone_plus_rail_global', 'weather_zone_plus_border_global', 'full_disruption_variable_only'), checkpoint_metrics=('best_val_pinball', 'best_val_q75', 'best_val_iqr'), overwrite=True, save_models=True, make_plots=True


## 4. Path management

This cell defines all input and output paths. The notebook reads:

- full PyG `HeteroData` from the topology build step,
- OD-year topology features,
- the disruption-enhanced supervised table,
- disruption feature QA and manifest files,
- previous Cold-OD and GraphV2 predictions for combined leaderboard comparison.


In [4]:

# ---------------------------------------------------------------------
# Path management.
# ---------------------------------------------------------------------

@dataclass
class ProjectPaths:
    data_root: Path
    scope: str
    run_name: str

    @property
    def supervised_path(self) -> Path:
        return self.data_root / "08_processed" / "model_ready" / f"freight_mnet_supervised_edges_2018_2024_{self.scope}.parquet"

    @property
    def feature_manifest_path(self) -> Path:
        return self.data_root / "08_processed" / "model_ready" / "_metadata" / "freight_mnet_supervised_feature_manifest.json"

    @property
    def heterodata_path(self) -> Path:
        return self.data_root / "08_processed" / "graph_inputs" / f"freight_mnet_full_heterodata_{self.scope}.pt"

    @property
    def topology_features_path(self) -> Path:
        return self.data_root / "08_processed" / "graph_inputs" / f"topology_features_od_{self.scope}.parquet"

    @property
    def disruption_dir(self) -> Path:
        return self.data_root / "08_processed" / "disruption_features"

    @property
    def supervised_with_disruption_path(self) -> Path:
        return self.disruption_dir / f"freight_mnet_supervised_edges_2018_2024_{self.scope}_with_disruption.parquet"

    @property
    def od_year_disruption_path(self) -> Path:
        return self.disruption_dir / f"disruption_features_od_year_{self.scope}.parquet"

    @property
    def disruption_manifest_path(self) -> Path:
        return self.disruption_dir / f"disruption_feature_manifest_{self.scope}.json"

    @property
    def disruption_qa_summary_path(self) -> Path:
        return self.data_root / "10_experiments" / "build_disruption_features_od_year_v1" / self.scope / "tables" / "disruption_feature_qa_summary.csv"

    @property
    def cold_baseline_predictions_path(self) -> Path:
        return self.data_root / "10_experiments" / "cold_od_split_baselines_v1_notebook" / self.scope / "predictions_cold_od_val_test.parquet"

    @property
    def graphv2_predictions_path(self) -> Path:
        return self.data_root / "10_experiments" / "graphsage_hgt_cold_od_baselines_v2_notebook" / self.scope / "combined_predictions_graph_cold_od_val_test_v2.parquet"

    @property
    def topology_v3_predictions_path(self) -> Path:
        return self.data_root / "10_experiments" / "dcqhgt_topology_cold_od_v3_ablation_notebook" / self.scope / "combined_predictions_dcqhgt_topology_v3_ablation_val_test.parquet"

    @property
    def output_dir(self) -> Path:
        return self.data_root / "10_experiments" / self.run_name / self.scope

    @property
    def tables_dir(self) -> Path:
        return self.output_dir / "tables"

    @property
    def models_dir(self) -> Path:
        return self.output_dir / "models"

    @property
    def plots_dir(self) -> Path:
        return self.output_dir / "plots"

    @property
    def reports_dir(self) -> Path:
        return self.output_dir / "reports"

paths = ProjectPaths(cfg.data_root, cfg.scope, cfg.run_name)

for directory in [paths.output_dir, paths.tables_dir, paths.models_dir, paths.plots_dir, paths.reports_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("Output directory:", paths.output_dir)


Output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\dcqhgt_disruption_cold_od_v1_notebook\east_plus_gulf



## 5. General utility functions

This cell defines utility functions for artifact checks, safe Parquet writing, JSON writing, reproducibility, and table display. These utilities are reused throughout the notebook.


In [5]:

# ---------------------------------------------------------------------
# General utility functions.
# ---------------------------------------------------------------------

LABEL_COLUMNS = ["truck_q25", "truck_q50", "truck_q75"]
TAUS = torch.tensor([0.25, 0.50, 0.75], dtype=torch.float32)

FORCE_STRING_COLUMNS = {
    "source", "model", "variant", "checkpoint", "checkpoint_metric", "seed", "split",
    "faf_orig", "faf_dest", "faf_orig_str", "faf_dest_str", "reference_name",
    "candidate_source", "candidate_model", "candidate_checkpoint_metric", "candidate_seed",
}


def ensure_file_exists(path: Path, description: str) -> None:
    """Raise a clear error if an expected file is missing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{description} not found: {path}")


def set_global_seed(seed: int) -> None:
    """Set Python, NumPy, and PyTorch random seeds."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_default(value):
    """Convert common scientific Python objects to JSON-safe values."""
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return str(value)


def write_json(payload: dict, path: Path) -> Path:
    """Write a JSON artifact."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, default=json_default)
    return path


def stringify_object_value(value):
    """Convert a scalar object into a Parquet-safe string or missing value."""
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, sort_keys=True, default=json_default)
    if value is None:
        return pd.NA
    try:
        if pd.isna(value):
            return pd.NA
    except Exception:
        pass
    return str(value)


def normalize_dataframe_for_parquet(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with object-like columns converted to string dtype."""
    clean = frame.copy()
    clean.columns = [str(col) for col in clean.columns]
    for col in clean.columns:
        series = clean[col]
        if col in FORCE_STRING_COLUMNS or pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
            clean[col] = series.map(stringify_object_value).astype("string")
    return clean


def safe_to_parquet(frame: pd.DataFrame, path: Path) -> Path:
    """Write a DataFrame to Parquet after dtype normalization."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    normalize_dataframe_for_parquet(frame).to_parquet(path, index=False, engine="pyarrow")
    return path


def log_frame(title: str, frame: pd.DataFrame, max_rows: int = 5) -> None:
    """Print a compact dataframe preview without requiring tabulate."""
    print(f"\n{title}: shape={frame.shape}")
    if frame.empty:
        print("  <empty>")
    else:
        print(frame.head(max_rows).to_string(index=False))


def dataframe_to_text(frame: pd.DataFrame, columns: Optional[Sequence[str]] = None, max_rows: int = 20) -> str:
    """Convert a dataframe slice to plain text without optional dependencies."""
    if frame.empty:
        return "_No rows available._"
    if columns is not None:
        columns = [col for col in columns if col in frame.columns]
        frame = frame.loc[:, columns]
    return frame.head(max_rows).to_string(index=False)



## 6. Load full HeteroData, supervised table, topology features, and disruption table

This cell loads all core data artifacts. The `HeteroData` object contains message-passing graph structure and supervised OD labels. The disruption-enhanced supervised table contains candidate disruption features generated in the previous notebook.


In [6]:

# ---------------------------------------------------------------------
# Load core data artifacts.
# ---------------------------------------------------------------------

ensure_file_exists(paths.heterodata_path, "Full topology HeteroData")
ensure_file_exists(paths.supervised_path, "Supervised model-ready table")
ensure_file_exists(paths.feature_manifest_path, "Feature manifest")
ensure_file_exists(paths.topology_features_path, "Topology feature table")
ensure_file_exists(paths.supervised_with_disruption_path, "Supervised table with disruption features")
ensure_file_exists(paths.disruption_qa_summary_path, "Disruption feature QA summary")

try:
    hetero_data = torch.load(paths.heterodata_path, map_location="cpu", weights_only=False)
except TypeError:
    hetero_data = torch.load(paths.heterodata_path, map_location="cpu")

if not isinstance(hetero_data, HeteroData):
    raise TypeError(f"Expected HeteroData, got {type(hetero_data)}")

supervised_df = pd.read_parquet(paths.supervised_path)
supervised_disruption_df = pd.read_parquet(paths.supervised_with_disruption_path)
topology_df = pd.read_parquet(paths.topology_features_path)
disruption_qa = pd.read_csv(paths.disruption_qa_summary_path)

with paths.feature_manifest_path.open("r", encoding="utf-8") as file:
    feature_manifest = json.load(file)
manifest_feature_columns = list(feature_manifest.get("feature_columns", []))
if not manifest_feature_columns:
    raise ValueError("Feature manifest does not contain feature_columns.")

label_store = hetero_data[("faf_zone", "supervised_od", "faf_zone")]

print("Loaded HeteroData:", hetero_data)
print("Supervised table:", supervised_df.shape)
print("Supervised + disruption table:", supervised_disruption_df.shape)
print("Topology table:", topology_df.shape)
print("Disruption QA summary:", disruption_qa.shape)
print("Manifest feature count:", len(manifest_feature_columns))

# Basic row alignment checks.
if len(supervised_df) != label_store.y.shape[0]:
    raise ValueError("Supervised table row count does not match HeteroData labels.")
if len(supervised_disruption_df) != label_store.y.shape[0]:
    raise ValueError("Supervised disruption table row count does not match HeteroData labels.")

log_frame("Disruption QA preview", disruption_qa)


Loaded HeteroData: HeteroData(
  faf_zone={
    x=[132, 86],
    node_id=[132],
  },
  terminal={
    x=[241, 5],
    node_id=[241],
  },
  (faf_zone, truck_adj, faf_zone)={
    edge_index=[2, 582],
    edge_attr=[582, 3],
  },
  (faf_zone, rail_adj, faf_zone)={
    edge_index=[2, 510],
    edge_attr=[510, 3],
  },
  (faf_zone, demand_truck, faf_zone)={
    edge_index=[2, 8693],
    edge_attr=[8693, 4],
  },
  (faf_zone, rev_demand_truck, faf_zone)={
    edge_index=[2, 8693],
    edge_attr=[8693, 4],
  },
  (faf_zone, demand_rail, faf_zone)={
    edge_index=[2, 5834],
    edge_attr=[5834, 4],
  },
  (faf_zone, rev_demand_rail, faf_zone)={
    edge_index=[2, 5834],
    edge_attr=[5834, 4],
  },
  (faf_zone, demand_multimodal, faf_zone)={
    edge_index=[2, 8692],
    edge_attr=[8692, 4],
  },
  (faf_zone, rev_demand_multimodal, faf_zone)={
    edge_index=[2, 8692],
    edge_attr=[8692, 4],
  },
  (faf_zone, train_od, faf_zone)={
    edge_index=[2, 8748],
    edge_attr=[8748, 2],
  },
  


## 7. Feature pruning and disruption group construction

This section implements the first required task: **feature pruning and disruption group ablation**.

The raw disruption table contains useful predictive features and also QA/source-availability columns. We remove zero-variance, all-zero, missing-heavy, and source-QA features. Then we construct interpretable groups:

- `weather_zone_only`
- `weather_global_only`
- `weather_all`
- `rail_global_only`
- `border_global_only`
- `weather_zone_plus_rail_global`
- `weather_zone_plus_border_global`
- `full_disruption_variable_only`


In [7]:

# ---------------------------------------------------------------------
# Feature pruning and disruption group construction.
# ---------------------------------------------------------------------

EXCLUDE_SUBSTRINGS = (
    "source_file_count",
    "source_missing",
    "feature_available",
    "_file_count",
)


def infer_qa_column(frame: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
    """Return the first candidate column present in a QA dataframe."""
    for col in candidates:
        if col in frame.columns:
            return col
    return None


def classify_disruption_feature(name: str) -> str:
    """Assign a disruption feature to a semantic group."""
    lower = name.lower()
    if "weather" in lower:
        if "global" in lower:
            return "weather_global"
        return "weather_zone"
    if "rail" in lower:
        if "global" in lower:
            return "rail_global"
        return "rail_zone"
    if "border" in lower:
        if "global" in lower:
            return "border_global"
        return "border_zone"
    return "other"


def build_clean_disruption_manifest(
    qa: pd.DataFrame,
    data_frame: pd.DataFrame,
    cfg: ExperimentConfig,
) -> Tuple[pd.DataFrame, Dict[str, List[str]]]:
    """Prune raw disruption features and build feature groups."""
    feature_col = infer_qa_column(qa, ["feature", "feature_name", "column", "column_name"])
    if feature_col is None:
        raise ValueError(f"Cannot infer feature-name column from QA summary columns: {list(qa.columns)}")

    missing_col = infer_qa_column(qa, ["missing_rate", "missing_fraction", "na_rate"])
    nonzero_col = infer_qa_column(qa, ["nonzero_rate", "non_zero_rate", "nonzero_fraction"])
    std_col = infer_qa_column(qa, ["std", "std_value", "standard_deviation"])

    rows = []
    for _, row in qa.iterrows():
        feature = str(row[feature_col])
        if feature not in data_frame.columns:
            continue

        missing_rate = float(row[missing_col]) if missing_col is not None and pd.notna(row[missing_col]) else float(data_frame[feature].isna().mean())
        nonzero_rate = float(row[nonzero_col]) if nonzero_col is not None and pd.notna(row[nonzero_col]) else float((pd.to_numeric(data_frame[feature], errors="coerce").fillna(0) != 0).mean())
        std_value = float(row[std_col]) if std_col is not None and pd.notna(row[std_col]) else float(pd.to_numeric(data_frame[feature], errors="coerce").std(skipna=True) or 0.0)

        lower = feature.lower()
        excluded_by_name = any(token in lower for token in EXCLUDE_SUBSTRINGS)
        keep = (
            missing_rate <= cfg.max_missing_rate
            and std_value > cfg.min_std
            and nonzero_rate > cfg.min_nonzero_rate
            and not excluded_by_name
        )

        rows.append(
            {
                "feature": feature,
                "group": classify_disruption_feature(feature),
                "missing_rate": missing_rate,
                "nonzero_rate": nonzero_rate,
                "std": std_value,
                "excluded_by_name": excluded_by_name,
                "keep": bool(keep),
            }
        )

    audit = pd.DataFrame(rows).sort_values(["keep", "group", "feature"], ascending=[False, True, True])

    kept = audit.loc[audit["keep"], "feature"].tolist()
    groups = {
        "no_disruption": [],
        "weather_zone_only": audit.query("keep and group == 'weather_zone'")["feature"].tolist(),
        "weather_global_only": audit.query("keep and group == 'weather_global'")["feature"].tolist(),
        "weather_all": audit.query("keep and group in ['weather_zone', 'weather_global']")["feature"].tolist(),
        "rail_global_only": audit.query("keep and group == 'rail_global'")["feature"].tolist(),
        "border_global_only": audit.query("keep and group == 'border_global'")["feature"].tolist(),
        "weather_zone_plus_rail_global": audit.query("keep and group in ['weather_zone', 'rail_global']")["feature"].tolist(),
        "weather_zone_plus_border_global": audit.query("keep and group in ['weather_zone', 'border_global']")["feature"].tolist(),
        "full_disruption_variable_only": kept,
    }

    # Keep only groups that are explicitly enabled and have at least one column, except no_disruption.
    groups = {
        name: columns
        for name, columns in groups.items()
        if name in cfg.enabled_disruption_groups and (name == "no_disruption" or len(columns) > 0)
    }

    return audit, groups


feature_pruning_audit, disruption_groups = build_clean_disruption_manifest(
    disruption_qa,
    supervised_disruption_df,
    cfg,
)

print("Kept disruption features:", int(feature_pruning_audit["keep"].sum()))
print("Dropped disruption features:", int((~feature_pruning_audit["keep"]).sum()))
print("\nDisruption groups:")
for group_name, columns in disruption_groups.items():
    print(f"  {group_name}: {len(columns)} columns")

feature_pruning_audit.to_csv(paths.tables_dir / "disruption_feature_pruning_audit.csv", index=False)
write_json(disruption_groups, paths.output_dir / "disruption_group_manifest.json")
log_frame("Feature pruning audit", feature_pruning_audit, max_rows=20)


Kept disruption features: 82
Dropped disruption features: 41

Disruption groups:
  no_disruption: 0 columns
  weather_zone_only: 60 columns
  weather_global_only: 12 columns
  weather_all: 72 columns
  rail_global_only: 4 columns
  border_global_only: 6 columns
  weather_zone_plus_rail_global: 64 columns
  weather_zone_plus_border_global: 66 columns
  full_disruption_variable_only: 82 columns

Feature pruning audit: shape=(123, 7)
                                           feature          group  missing_rate  nonzero_rate          std  excluded_by_name  keep
disrupt_border_global_canada_truck_crossings_total  border_global           0.0      1.000000 1.240893e+06             False  True
 disrupt_border_global_log1p_truck_crossings_total  border_global           0.0      1.000000 3.847867e-02             False  True
disrupt_border_global_mexico_truck_crossings_total  border_global           0.0      1.000000 3.468456e+06             False  True
        disrupt_border_global_truck_cross


## 8. Graph schema utilities and bidirectional topology check

This cell prepares edge-type subsets for the two stable topology candidates and ensures truck / rail topology relations are bidirectional. This makes the topology candidates consistent with the v2 topology experiment.


In [8]:

# ---------------------------------------------------------------------
# Graph schema utilities and bidirectional topology check.
# ---------------------------------------------------------------------

FAF = "faf_zone"
TERM = "terminal"
SUP_EDGE = (FAF, "supervised_od", FAF)

BASE_FAF_EDGE_TYPES = [
    (FAF, "demand_truck", FAF),
    (FAF, "rev_demand_truck", FAF),
    (FAF, "demand_rail", FAF),
    (FAF, "rev_demand_rail", FAF),
    (FAF, "demand_multimodal", FAF),
    (FAF, "rev_demand_multimodal", FAF),
    (FAF, "train_od", FAF),
    (FAF, "rev_train_od", FAF),
    (FAF, "self_loop", FAF),
]
TRUCK_RAIL_EDGE_TYPES = [(FAF, "truck_adj", FAF), (FAF, "rail_adj", FAF)]
TERMINAL_EDGE_TYPES = [(FAF, "terminal_access", TERM), (TERM, "reverse_terminal_access", FAF)]


@dataclass(frozen=True)
class TopologyCandidate:
    name: str
    edge_types: Tuple[Tuple[str, str, str], ...]
    use_terminal_nodes: bool
    use_topology_decoder_features: bool
    description: str


TOPOLOGY_CANDIDATES: Dict[str, TopologyCandidate] = {
    "hgt_truck_rail_plus_topology_features": TopologyCandidate(
        name="hgt_truck_rail_plus_topology_features",
        edge_types=tuple(BASE_FAF_EDGE_TYPES + TRUCK_RAIL_EDGE_TYPES),
        use_terminal_nodes=False,
        use_topology_decoder_features=True,
        description="Overall candidate: truck/rail topology message passing plus topology/path decoder features.",
    ),
    "hgt_terminal_plus_truck_plus_rail": TopologyCandidate(
        name="hgt_terminal_plus_truck_plus_rail",
        edge_types=tuple(BASE_FAF_EDGE_TYPES + TRUCK_RAIL_EDGE_TYPES + TERMINAL_EDGE_TYPES),
        use_terminal_nodes=True,
        use_topology_decoder_features=False,
        description="Sparse/risk candidate: terminal access plus truck/rail topology message passing.",
    ),
}


def edge_storage_has(data: HeteroData, edge_type: Tuple[str, str, str], key: str) -> bool:
    """Return True if a PyG edge storage contains a key."""
    return edge_type in data.edge_types and key in data[edge_type].keys()


def bidirectional_ratio(edge_index: torch.Tensor) -> Tuple[int, int, float]:
    """Return count and ratio of edges whose reverse edge is present."""
    pairs = set(map(tuple, edge_index.t().cpu().numpy().tolist()))
    reverse_pairs = {(dst, src) for src, dst in pairs}
    both = pairs & reverse_pairs
    return len(both), len(pairs), len(both) / max(len(pairs), 1)


def make_edge_type_bidirectional_in_place(data: HeteroData, edge_type: Tuple[str, str, str]) -> dict:
    """Ensure a homogeneous edge type contains both directions within the same relation."""
    if edge_type not in data.edge_types or "edge_index" not in data[edge_type].keys():
        return {"edge_type": str(edge_type), "status": "missing"}

    store = data[edge_type]
    edge_index = store.edge_index.cpu()
    edge_attr = store.edge_attr.cpu() if "edge_attr" in store.keys() else None

    before_both, before_total, before_ratio = bidirectional_ratio(edge_index)
    pairs = list(map(tuple, edge_index.t().numpy().tolist()))
    pair_set = set(pairs)

    new_edges = []
    new_attrs = []
    if edge_attr is not None:
        attr_rows = edge_attr.numpy()
    else:
        attr_rows = [None] * len(pairs)

    for idx, (src, dst) in enumerate(pairs):
        rev = (dst, src)
        if rev not in pair_set:
            new_edges.append(rev)
            if edge_attr is not None:
                new_attrs.append(attr_rows[idx])

    if new_edges:
        add_index = torch.tensor(new_edges, dtype=edge_index.dtype).t().contiguous()
        store.edge_index = torch.cat([edge_index, add_index], dim=1)
        if edge_attr is not None:
            add_attr = torch.tensor(np.asarray(new_attrs), dtype=edge_attr.dtype)
            store.edge_attr = torch.cat([edge_attr, add_attr], dim=0)

    after_both, after_total, after_ratio = bidirectional_ratio(store.edge_index.cpu())
    return {
        "edge_type": str(edge_type),
        "before_edges": before_total,
        "before_bidirectional_ratio": before_ratio,
        "added_reverse_edges": len(new_edges),
        "after_edges": after_total,
        "after_bidirectional_ratio": after_ratio,
    }


repair_rows = []
for topology_edge_type in TRUCK_RAIL_EDGE_TYPES:
    repair_rows.append(make_edge_type_bidirectional_in_place(hetero_data, topology_edge_type))

repair_report = pd.DataFrame(repair_rows)
repair_report.to_csv(paths.tables_dir / "disruption_topology_bidirectional_repair_report.csv", index=False)
log_frame("Bidirectional topology repair report", repair_report)



Bidirectional topology repair report: shape=(2, 6)
                            edge_type  before_edges  before_bidirectional_ratio  added_reverse_edges  after_edges  after_bidirectional_ratio
('faf_zone', 'truck_adj', 'faf_zone')           582                         1.0                    0          582                        1.0
 ('faf_zone', 'rail_adj', 'faf_zone')           510                         1.0                    0          510                        1.0



## 9. Tensor preparation

This cell extracts tensors from the supervised OD edge store, builds train-only sample weights, constructs Cold-OD fallback priors, and creates edge feature matrices for each topology candidate.

The model uses a residualized monotone quantile head around a train-only fallback prior. This keeps the Cold-OD setup valid because exact test OD-pair history is not used.


In [9]:

# ---------------------------------------------------------------------
# Tensor preparation.
# ---------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

store = hetero_data[SUP_EDGE]
y_all = store.y.float()
edge_label_index_all = store.edge_label_index.long()
row_id_all = store.row_id.long() if "row_id" in store.keys() else torch.arange(y_all.shape[0], dtype=torch.long)
year_all = store.year.long() if "year" in store.keys() else torch.tensor(supervised_disruption_df["year"].to_numpy(), dtype=torch.long)

cold_train_mask = store.cold_train_mask.bool()
cold_val_mask = store.cold_val_mask.bool()
cold_test_mask = store.cold_test_mask.bool()

# Edge features: first 64 columns are the original manifest features, remaining columns are topology/path features.
edge_attr_raw_all = store.edge_label_attr_raw.float() if "edge_label_attr_raw" in store.keys() else store.edge_label_attr.float()
current_feature_dim = len(manifest_feature_columns)
if edge_attr_raw_all.shape[1] < current_feature_dim:
    raise ValueError("HeteroData edge_label_attr has fewer columns than the feature manifest.")

def candidate_edge_features(candidate: TopologyCandidate) -> torch.Tensor:
    """Return raw supervised edge features for a topology candidate."""
    if candidate.use_topology_decoder_features:
        return edge_attr_raw_all.clone()
    return edge_attr_raw_all[:, :current_feature_dim].clone()


def compute_sample_weights(frame: pd.DataFrame, train_mask: torch.Tensor, column: str) -> torch.Tensor:
    """Build normalized sample weights from obs_weight_sum or fallback to ones."""
    if column not in frame.columns:
        return torch.ones(len(frame), dtype=torch.float32)

    raw = pd.to_numeric(frame[column], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
    weights = np.log1p(np.maximum(raw, 0.0))
    train_values = weights[train_mask.cpu().numpy()]
    mean_train = float(np.nanmean(train_values)) if len(train_values) else 1.0
    if not np.isfinite(mean_train) or mean_train <= 0:
        mean_train = 1.0
    weights = weights / mean_train
    weights = np.clip(weights, cfg.weight_clip_min, cfg.weight_clip_max)
    return torch.tensor(weights, dtype=torch.float32)

sample_weight_all = compute_sample_weights(supervised_disruption_df, cold_train_mask, cfg.weight_column)

def compute_zone_fallback_prior(y: torch.Tensor, edge_index: torch.Tensor, train_mask: torch.Tensor) -> torch.Tensor:
    """Compute origin/destination blend fallback quantile prior from training rows only."""
    src = edge_index[0].cpu().numpy()
    dst = edge_index[1].cpu().numpy()
    train_idx = torch.where(train_mask.cpu())[0].numpy()
    y_np = y.cpu().numpy()
    n_rows = y_np.shape[0]

    global_prior = np.median(y_np[train_idx], axis=0)

    origin_values: Dict[int, List[np.ndarray]] = {}
    dest_values: Dict[int, List[np.ndarray]] = {}
    for idx in train_idx:
        origin_values.setdefault(int(src[idx]), []).append(y_np[idx])
        dest_values.setdefault(int(dst[idx]), []).append(y_np[idx])

    origin_prior = {key: np.median(np.vstack(vals), axis=0) for key, vals in origin_values.items()}
    dest_prior = {key: np.median(np.vstack(vals), axis=0) for key, vals in dest_values.items()}

    base = np.zeros((n_rows, 3), dtype=np.float32)
    for i in range(n_rows):
        parts = []
        if int(src[i]) in origin_prior:
            parts.append(origin_prior[int(src[i])])
        if int(dst[i]) in dest_prior:
            parts.append(dest_prior[int(dst[i])])
        if parts:
            base[i] = np.mean(np.vstack(parts), axis=0)
        else:
            base[i] = global_prior

    base[:, 0] = np.maximum(base[:, 0], 1e-3)
    base[:, 1] = np.maximum(base[:, 1], base[:, 0] + 1e-3)
    base[:, 2] = np.maximum(base[:, 2], base[:, 1] + 1e-3)
    return torch.tensor(base, dtype=torch.float32)

base_prior_all = compute_zone_fallback_prior(y_all, edge_label_index_all, cold_train_mask)
target_scale = float(torch.median(y_all[cold_train_mask, 1]).item())
if not np.isfinite(target_scale) or target_scale <= 0:
    target_scale = 1.0

# Year index is compact 0..n_years-1 for embedding.
years_sorted = sorted(pd.Series(year_all.cpu().numpy()).dropna().astype(int).unique().tolist())
year_to_idx = {year: idx for idx, year in enumerate(years_sorted)}
year_idx_all = torch.tensor([year_to_idx[int(year)] for year in year_all.cpu().numpy()], dtype=torch.long)

print("Target scale:", target_scale)
print("Years:", year_to_idx)
print("Cold split counts:", {
    "train": int(cold_train_mask.sum()),
    "val": int(cold_val_mask.sum()),
    "test": int(cold_test_mask.sum()),
})


Device: cuda
Target scale: 2317.030029296875
Years: {2018: 0, 2019: 1, 2020: 2, 2021: 3, 2022: 4, 2023: 5, 2024: 6}
Cold split counts: {'train': 42849, 'val': 957, 'test': 1057}



## 10. Numeric preprocessing utilities

This section defines train-only median imputation and z-score scaling. Separate preprocessors are fitted for each edge-feature matrix and each disruption group, using only Cold-OD training rows.


In [10]:

# ---------------------------------------------------------------------
# Numeric preprocessing utilities.
# ---------------------------------------------------------------------

@dataclass
class NumericPreprocessor:
    median: List[float]
    mean: List[float]
    std: List[float]
    columns: List[str]

    def transform_array(self, values: np.ndarray) -> np.ndarray:
        """Apply median imputation and z-score scaling."""
        x = values.astype(np.float32, copy=True)
        med = np.asarray(self.median, dtype=np.float32)
        mean = np.asarray(self.mean, dtype=np.float32)
        std = np.asarray(self.std, dtype=np.float32)
        invalid = ~np.isfinite(x)
        if invalid.any():
            x[invalid] = np.take(med, np.where(invalid)[1])
        return (x - mean) / std

    def transform_tensor(self, values: torch.Tensor) -> torch.Tensor:
        arr = values.detach().cpu().numpy()
        return torch.tensor(self.transform_array(arr), dtype=torch.float32)

    def to_dict(self) -> dict:
        return asdict(self)


def fit_numeric_preprocessor(values: np.ndarray, train_mask: np.ndarray, columns: Sequence[str]) -> NumericPreprocessor:
    """Fit a robust numeric preprocessor on training rows only."""
    train = values[train_mask].astype(np.float32)
    train = np.where(np.isfinite(train), train, np.nan)
    median = np.nanmedian(train, axis=0)
    median = np.where(np.isfinite(median), median, 0.0)
    filled = np.where(np.isfinite(train), train, median)
    mean = filled.mean(axis=0)
    std = filled.std(axis=0)
    std = np.where(np.isfinite(std) & (std > 1e-6), std, 1.0)
    return NumericPreprocessor(median=median.tolist(), mean=mean.tolist(), std=std.tolist(), columns=list(columns))


def build_disruption_matrix(columns: Sequence[str]) -> np.ndarray:
    """Return a numeric disruption feature matrix for selected columns."""
    if len(columns) == 0:
        return np.zeros((len(supervised_disruption_df), 0), dtype=np.float32)
    matrix = supervised_disruption_df.loc[:, list(columns)].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    return matrix

train_mask_np = cold_train_mask.cpu().numpy()



## 11. Model definitions

The D-CQHGT disruption model uses:

- HGTConv for typed graph message passing,
- an edge feature MLP,
- a disruption MLP and sigmoid gate,
- a residualized monotone quantile head around the Cold-OD fallback prior.

The gate is applied to the edge hidden representation so that disruption exposure can modulate q75 and IQR behavior.


In [11]:

# ---------------------------------------------------------------------
# Model definitions.
# ---------------------------------------------------------------------

class MLP(nn.Module):
    """Small feed-forward network with GELU activations."""
    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout: float = 0.1):
        super().__init__()
        if in_dim == 0:
            self.net = None
            self.out_dim = out_dim
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim),
                nn.GELU(),
            )
            self.out_dim = out_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.net is None:
            return torch.zeros((x.shape[0], self.out_dim), device=x.device, dtype=x.dtype)
        return self.net(x)


def inverse_softplus(x: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    """Numerically stable inverse softplus for positive tensors."""
    x = torch.clamp(x, min=eps)
    return torch.log(torch.expm1(torch.clamp(x, max=20.0)))


class DisruptionCalibratedHGT(nn.Module):
    """HGT edge predictor with disruption gate and residual monotone quantile head."""
    def __init__(
        self,
        metadata,
        node_input_dims: Dict[str, int],
        edge_feature_dim: int,
        disruption_dim: int,
        num_years: int,
        hidden_dim: int = 128,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.1,
        gate_alpha: float = 1.0,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.gate_alpha = gate_alpha
        self.node_proj = nn.ModuleDict({
            ntype: nn.Linear(in_dim, hidden_dim)
            for ntype, in_dim in node_input_dims.items()
        })
        self.year_emb = nn.Embedding(num_years, hidden_dim)
        self.convs = nn.ModuleList([
            HGTConv(hidden_dim, hidden_dim, metadata, heads=num_heads)
            for _ in range(num_layers)
        ])
        self.edge_mlp = MLP(edge_feature_dim, hidden_dim, hidden_dim, dropout)
        self.disruption_mlp = MLP(disruption_dim, hidden_dim, hidden_dim, dropout)
        decoder_dim = hidden_dim * 6
        self.decoder = nn.Sequential(
            nn.Linear(decoder_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
        )
        self.out = nn.Linear(hidden_dim, 3)

    def forward(
        self,
        x_dict: Dict[str, torch.Tensor],
        edge_index_dict: Dict[Tuple[str, str, str], torch.Tensor],
        edge_label_index: torch.Tensor,
        edge_attr: torch.Tensor,
        disruption_attr: torch.Tensor,
        base_prior: torch.Tensor,
        year_idx: torch.Tensor,
        target_scale: float,
    ) -> torch.Tensor:
        """Predict q25/q50/q75 for a batch of supervised OD edges."""
        h_dict = {ntype: self.node_proj[ntype](x) for ntype, x in x_dict.items()}
        for conv in self.convs:
            h_dict = conv(h_dict, edge_index_dict)
            h_dict = {key: F.gelu(value) for key, value in h_dict.items()}

        src, dst = edge_label_index
        u = h_dict[FAF][src]
        v = h_dict[FAF][dst]
        e = self.edge_mlp(edge_attr)
        d = self.disruption_mlp(disruption_attr)
        gate = torch.sigmoid(d)
        e_gated = e * (1.0 + self.gate_alpha * gate)
        yemb = self.year_emb(year_idx)

        z = torch.cat([u, v, torch.abs(u - v), u * v, e_gated, yemb], dim=-1)
        raw = self.out(self.decoder(z))

        base_scaled = base_prior / float(target_scale)
        base_q25 = torch.clamp(base_scaled[:, 0], min=1e-4)
        base_gap50 = torch.clamp(base_scaled[:, 1] - base_scaled[:, 0], min=1e-4)
        base_gap75 = torch.clamp(base_scaled[:, 2] - base_scaled[:, 1], min=1e-4)

        q25_s = F.softplus(inverse_softplus(base_q25) + raw[:, 0])
        gap50_s = F.softplus(inverse_softplus(base_gap50) + raw[:, 1])
        gap75_s = F.softplus(inverse_softplus(base_gap75) + raw[:, 2])
        q50_s = q25_s + gap50_s
        q75_s = q50_s + gap75_s

        pred_scaled = torch.stack([q25_s, q50_s, q75_s], dim=-1)
        return pred_scaled * float(target_scale)



## 12. Losses and metrics

The training objective is weighted pinball loss with an IQR auxiliary loss. Evaluation reports q25/q50/q75 MAE, pinball losses, IQR MAE, stress q75, top-IQR, and sparse-label metrics.


In [12]:

# ---------------------------------------------------------------------
# Losses and metrics.
# ---------------------------------------------------------------------

TAUS_CPU = np.array([0.25, 0.50, 0.75], dtype=np.float32)


def pinball_loss_tensor(pred: torch.Tensor, target: torch.Tensor, taus: torch.Tensor) -> torch.Tensor:
    """Return per-row mean pinball loss."""
    taus = taus.to(pred.device).view(1, -1)
    diff = target - pred
    loss = torch.maximum(taus * diff, (taus - 1.0) * diff)
    return loss.mean(dim=1)


def training_loss(pred: torch.Tensor, target: torch.Tensor, weight: torch.Tensor, lambda_iqr: float) -> torch.Tensor:
    """Weighted pinball loss plus IQR auxiliary loss."""
    row_pinball = pinball_loss_tensor(pred, target, TAUS.to(pred.device))
    weighted = (row_pinball * weight).mean()
    pred_iqr = pred[:, 2] - pred[:, 0]
    true_iqr = target[:, 2] - target[:, 0]
    iqr_loss = F.smooth_l1_loss(pred_iqr, true_iqr)
    return weighted + lambda_iqr * iqr_loss


def compute_metrics_from_arrays(frame: pd.DataFrame) -> dict:
    """Compute evaluation metrics from a prediction dataframe."""
    y = frame[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float64)
    p = frame[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float64)
    diff = y - p
    abs_err = np.abs(diff)
    pin = np.maximum(TAUS_CPU.reshape(1, 3) * diff, (TAUS_CPU.reshape(1, 3) - 1.0) * diff)
    pred_iqr = p[:, 2] - p[:, 0]
    true_iqr = y[:, 2] - y[:, 0]

    metrics = {
        "n_rows": int(len(frame)),
        "mae_q25": float(abs_err[:, 0].mean()),
        "mae_q50": float(abs_err[:, 1].mean()),
        "mae_q75": float(abs_err[:, 2].mean()),
        "pinball_q25": float(pin[:, 0].mean()),
        "pinball_q50": float(pin[:, 1].mean()),
        "pinball_q75": float(pin[:, 2].mean()),
        "pinball_mean": float(pin.mean()),
        "iqr_mae": float(np.abs(pred_iqr - true_iqr).mean()),
        "bias_q25": float((p[:, 0] - y[:, 0]).mean()),
        "bias_q50": float((p[:, 1] - y[:, 1]).mean()),
        "bias_q75": float((p[:, 2] - y[:, 2]).mean()),
        "raw_crossing_rate": float(((p[:, 0] > p[:, 1]) | (p[:, 1] > p[:, 2])).mean()),
    }

    if "sample_weight" in frame.columns:
        w = frame["sample_weight"].to_numpy(dtype=np.float64)
        w = w / max(w.mean(), 1e-12)
        metrics["weighted_pinball_mean"] = float(np.average(pin.mean(axis=1), weights=w))
    else:
        metrics["weighted_pinball_mean"] = metrics["pinball_mean"]

    if len(frame) > 0:
        q75_threshold = np.quantile(y[:, 2], 0.90)
        stress_mask = y[:, 2] >= q75_threshold
        metrics["stress_top10_threshold_q75"] = float(q75_threshold)
        metrics["stress_top10_n"] = int(stress_mask.sum())
        metrics["stress_top10_mae_q75"] = float(abs_err[stress_mask, 2].mean()) if stress_mask.any() else np.nan

        iqr_threshold = np.quantile(true_iqr, 0.90)
        iqr_mask = true_iqr >= iqr_threshold
        metrics["top_iqr10_threshold"] = float(iqr_threshold)
        metrics["top_iqr10_n"] = int(iqr_mask.sum())
        metrics["top_iqr10_mae_q75"] = float(abs_err[iqr_mask, 2].mean()) if iqr_mask.any() else np.nan
        metrics["top_iqr10_iqr_mae"] = float(np.abs(pred_iqr[iqr_mask] - true_iqr[iqr_mask]).mean()) if iqr_mask.any() else np.nan

    if "n_fmi_county_pairs" in frame.columns:
        threshold = np.quantile(frame["n_fmi_county_pairs"].to_numpy(dtype=np.float64), 0.25)
        sparse_mask = frame["n_fmi_county_pairs"].to_numpy(dtype=np.float64) <= threshold
        metrics["sparse_bottom25_threshold_n_fmi"] = float(threshold)
        metrics["sparse_bottom25_n"] = int(sparse_mask.sum())
        metrics["sparse_bottom25_mae_q75"] = float(abs_err[sparse_mask, 2].mean()) if sparse_mask.any() else np.nan
        metrics["sparse_bottom25_iqr_mae"] = float(np.abs(pred_iqr[sparse_mask] - true_iqr[sparse_mask]).mean()) if sparse_mask.any() else np.nan

    return metrics



## 13. Build active graph inputs for each topology candidate

This cell builds candidate-specific `x_dict`, `edge_index_dict`, metadata, and node input dimensions. The overall candidate uses truck/rail topology plus topology decoder features. The sparse/risk candidate uses terminal access plus truck/rail topology.


In [13]:

# ---------------------------------------------------------------------
# Build active graph inputs for topology candidates.
# ---------------------------------------------------------------------


def build_candidate_graph_inputs(data: HeteroData, candidate: TopologyCandidate, device: torch.device):
    """Build x_dict, edge_index_dict, metadata, and input dims for a candidate."""
    active_node_types = {FAF}
    if candidate.use_terminal_nodes:
        active_node_types.add(TERM)

    x_dict = {
        ntype: data[ntype].x.float().to(device)
        for ntype in sorted(active_node_types)
    }

    edge_index_dict = {}
    for edge_type in candidate.edge_types:
        if edge_type in data.edge_types and "edge_index" in data[edge_type].keys():
            edge_index_dict[edge_type] = data[edge_type].edge_index.long().to(device)

    node_input_dims = {ntype: int(x.shape[1]) for ntype, x in x_dict.items()}
    metadata = (list(x_dict.keys()), list(edge_index_dict.keys()))
    return x_dict, edge_index_dict, metadata, node_input_dims


def build_edge_feature_preprocessor(candidate: TopologyCandidate) -> Tuple[NumericPreprocessor, torch.Tensor, List[str]]:
    """Fit train-only preprocessor and transform edge features for a candidate."""
    raw_tensor = candidate_edge_features(candidate)
    raw_np = raw_tensor.cpu().numpy().astype(np.float32)
    if candidate.use_topology_decoder_features:
        edge_cols = manifest_feature_columns + [f"topology_feature_{i}" for i in range(raw_np.shape[1] - current_feature_dim)]
    else:
        edge_cols = manifest_feature_columns
    preprocessor = fit_numeric_preprocessor(raw_np, train_mask_np, edge_cols)
    transformed = preprocessor.transform_tensor(raw_tensor)
    return preprocessor, transformed, edge_cols

candidate_graph_inputs = {}
edge_preprocessors = {}
edge_feature_tensors = {}
edge_feature_columns_by_candidate = {}

for candidate_name in cfg.enabled_topology_candidates:
    candidate = TOPOLOGY_CANDIDATES[candidate_name]
    candidate_graph_inputs[candidate_name] = build_candidate_graph_inputs(hetero_data, candidate, device)
    preprocessor, transformed, edge_cols = build_edge_feature_preprocessor(candidate)
    edge_preprocessors[candidate_name] = preprocessor
    edge_feature_tensors[candidate_name] = transformed.to(device)
    edge_feature_columns_by_candidate[candidate_name] = edge_cols
    print(candidate_name, "edge feature dim:", transformed.shape[1], "active edge types:", len(candidate_graph_inputs[candidate_name][1]))


C:\Users\Nick_James\AppData\Local\Temp\ipykernel_68188\2958298329.py:13: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  ntype: data[ntype].x.float().to(device)


hgt_truck_rail_plus_topology_features edge feature dim: 80 active edge types: 11
hgt_terminal_plus_truck_plus_rail edge feature dim: 64 active edge types: 13



## 14. Disruption group preprocessing

This cell builds group-specific disruption tensors. Each group is preprocessed using training rows only. The `no_disruption` group has zero columns and is handled by the model automatically.


In [14]:

# ---------------------------------------------------------------------
# Disruption group preprocessing.
# ---------------------------------------------------------------------

disruption_preprocessors = {}
disruption_tensors = {}

for group_name, columns in disruption_groups.items():
    matrix = build_disruption_matrix(columns)
    if matrix.shape[1] == 0:
        transformed = torch.zeros((len(supervised_disruption_df), 0), dtype=torch.float32)
        preprocessor = NumericPreprocessor(median=[], mean=[], std=[], columns=[])
    else:
        preprocessor = fit_numeric_preprocessor(matrix, train_mask_np, columns)
        transformed = torch.tensor(preprocessor.transform_array(matrix), dtype=torch.float32)
    disruption_preprocessors[group_name] = preprocessor
    disruption_tensors[group_name] = transformed.to(device)
    print(group_name, "dim:", transformed.shape[1])


no_disruption dim: 0
weather_zone_only dim: 60
weather_global_only dim: 12
weather_all dim: 72
rail_global_only dim: 4
border_global_only dim: 6
weather_zone_plus_rail_global dim: 64
weather_zone_plus_border_global dim: 66
full_disruption_variable_only dim: 82



## 15. Training and prediction utilities

This section defines reusable training functions. It trains each topology candidate × disruption group × seed, saves three checkpoints, and generates validation/test predictions.


In [15]:

# ---------------------------------------------------------------------
# Training and prediction utilities.
# ---------------------------------------------------------------------

indices = torch.arange(y_all.shape[0], dtype=torch.long)
train_indices = indices[cold_train_mask]
val_indices = indices[cold_val_mask]
test_indices = indices[cold_test_mask]

# Move always-used tensors to device.
y_all_dev = y_all.to(device)
edge_label_index_all_dev = edge_label_index_all.to(device)
base_prior_all_dev = base_prior_all.to(device)
sample_weight_all_dev = sample_weight_all.to(device)
year_idx_all_dev = year_idx_all.to(device)


def make_loader(row_indices: torch.Tensor, batch_size: int, shuffle: bool) -> DataLoader:
    """Create a DataLoader over row indices."""
    dataset = TensorDataset(row_indices.long())
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def evaluate_model_on_indices(
    model: DisruptionCalibratedHGT,
    candidate_name: str,
    disruption_group: str,
    row_indices: torch.Tensor,
) -> pd.DataFrame:
    """Run model prediction for selected rows and return a prediction dataframe."""
    candidate = TOPOLOGY_CANDIDATES[candidate_name]
    x_dict, edge_index_dict, _, _ = candidate_graph_inputs[candidate_name]
    edge_attr_all = edge_feature_tensors[candidate_name]
    disruption_attr_all = disruption_tensors[disruption_group]

    model.eval()
    preds = []
    idx_list = []
    loader = make_loader(row_indices.cpu(), cfg.batch_size, shuffle=False)
    with torch.no_grad():
        for (batch_idx_cpu,) in loader:
            batch_idx = batch_idx_cpu.to(device)
            batch_pred = model(
                x_dict=x_dict,
                edge_index_dict=edge_index_dict,
                edge_label_index=edge_label_index_all_dev[:, batch_idx],
                edge_attr=edge_attr_all[batch_idx],
                disruption_attr=disruption_attr_all[batch_idx],
                base_prior=base_prior_all_dev[batch_idx],
                year_idx=year_idx_all_dev[batch_idx],
                target_scale=target_scale,
            )
            preds.append(batch_pred.cpu())
            idx_list.append(batch_idx_cpu.cpu())

    pred = torch.cat(preds, dim=0).numpy()
    idx = torch.cat(idx_list, dim=0).numpy()
    true = y_all[idx].numpy()

    out = pd.DataFrame({
        "row_id": row_id_all[idx].numpy().astype(int),
        "split": np.where(cold_train_mask[idx].numpy(), "train", np.where(cold_val_mask[idx].numpy(), "val", "test")),
        "year": year_all[idx].numpy().astype(int),
        "true_q25": true[:, 0],
        "true_q50": true[:, 1],
        "true_q75": true[:, 2],
        "pred_q25": pred[:, 0],
        "pred_q50": pred[:, 1],
        "pred_q75": pred[:, 2],
        "sample_weight": sample_weight_all[idx].numpy(),
    })

    # Attach useful diagnostics for stress and sparse metrics.
    for col in ["n_fmi_county_pairs", "obs_weight_sum", "faf_orig", "faf_dest"]:
        if col in supervised_disruption_df.columns:
            out[col] = supervised_disruption_df.iloc[idx][col].to_numpy()

    return out


def train_one_run(candidate_name: str, disruption_group: str, seed: int) -> Tuple[List[dict], Dict[str, dict], Dict[str, nn.Module]]:
    """Train one candidate/group/seed and return history plus best states."""
    set_global_seed(seed)
    candidate = TOPOLOGY_CANDIDATES[candidate_name]
    x_dict, edge_index_dict, metadata, node_input_dims = candidate_graph_inputs[candidate_name]
    edge_attr_all = edge_feature_tensors[candidate_name]
    disruption_attr_all = disruption_tensors[disruption_group]

    model = DisruptionCalibratedHGT(
        metadata=metadata,
        node_input_dims=node_input_dims,
        edge_feature_dim=edge_attr_all.shape[1],
        disruption_dim=disruption_attr_all.shape[1],
        num_years=len(year_to_idx),
        hidden_dim=cfg.hidden_dim,
        num_layers=cfg.num_layers,
        num_heads=cfg.num_heads,
        dropout=cfg.dropout,
        gate_alpha=cfg.disruption_gate_alpha,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    train_loader = make_loader(train_indices.cpu(), cfg.batch_size, shuffle=True)

    best_scores = {metric: math.inf for metric in cfg.checkpoint_metrics}
    best_states = {}
    best_epoch = {metric: -1 for metric in cfg.checkpoint_metrics}
    history = []
    stale_epochs = 0

    for epoch in range(1, cfg.max_epochs + 1):
        model.train()
        epoch_losses = []
        for (batch_idx_cpu,) in train_loader:
            batch_idx = batch_idx_cpu.to(device)
            pred = model(
                x_dict=x_dict,
                edge_index_dict=edge_index_dict,
                edge_label_index=edge_label_index_all_dev[:, batch_idx],
                edge_attr=edge_attr_all[batch_idx],
                disruption_attr=disruption_attr_all[batch_idx],
                base_prior=base_prior_all_dev[batch_idx],
                year_idx=year_idx_all_dev[batch_idx],
                target_scale=target_scale,
            )
            loss = training_loss(pred, y_all_dev[batch_idx], sample_weight_all_dev[batch_idx], cfg.lambda_iqr)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            optimizer.step()
            epoch_losses.append(float(loss.detach().cpu()))

        val_pred = evaluate_model_on_indices(model, candidate_name, disruption_group, val_indices)
        val_metrics = compute_metrics_from_arrays(val_pred)
        current_scores = {
            "best_val_pinball": val_metrics["weighted_pinball_mean"],
            "best_val_q75": val_metrics["mae_q75"],
            "best_val_iqr": val_metrics["iqr_mae"],
        }

        improved = False
        for metric_name, score in current_scores.items():
            if score < best_scores[metric_name] - 1e-6:
                best_scores[metric_name] = score
                best_epoch[metric_name] = epoch
                best_states[metric_name] = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
                improved = True

        if improved:
            stale_epochs = 0
        else:
            stale_epochs += 1

        row = {
            "candidate": candidate_name,
            "disruption_group": disruption_group,
            "seed": seed,
            "epoch": epoch,
            "train_loss": float(np.mean(epoch_losses)) if epoch_losses else np.nan,
            **{f"val_{k}": v for k, v in val_metrics.items() if isinstance(v, (int, float, np.floating))},
        }
        history.append(row)

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"[{candidate_name} | {disruption_group} | seed {seed}] "
                f"epoch={epoch} train_loss={row['train_loss']:.4f} "
                f"val_wpin={val_metrics['weighted_pinball_mean']:.3f} "
                f"val_q75={val_metrics['mae_q75']:.3f} val_iqr={val_metrics['iqr_mae']:.3f}"
            )

        if stale_epochs >= cfg.patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    checkpoint_summary = {
        metric: {"score": float(best_scores[metric]), "epoch": int(best_epoch[metric])}
        for metric in cfg.checkpoint_metrics
    }

    return history, checkpoint_summary, best_states



## 16. Run disruption group ablation

This is the main training cell. It trains every enabled topology candidate, disruption group, and seed. The outputs are prediction tables, checkpoint summaries, and training histories.


In [16]:

# ---------------------------------------------------------------------
# Run disruption group ablation.
# ---------------------------------------------------------------------

all_history_rows = []
checkpoint_rows = []
prediction_tables = []

start_time = time.time()

for candidate_name in cfg.enabled_topology_candidates:
    for disruption_group in disruption_groups.keys():
        for seed in cfg.seeds:
            print("\n" + "=" * 90)
            print(f"Training candidate={candidate_name}, disruption_group={disruption_group}, seed={seed}")
            print("=" * 90)

            history, checkpoint_summary, best_states = train_one_run(candidate_name, disruption_group, seed)
            all_history_rows.extend(history)

            for metric_name, summary in checkpoint_summary.items():
                checkpoint_rows.append({
                    "candidate": candidate_name,
                    "disruption_group": disruption_group,
                    "seed": seed,
                    "checkpoint_metric": metric_name,
                    "best_epoch": summary["epoch"],
                    "best_score": summary["score"],
                })

                # Rebuild model for the saved checkpoint.
                candidate = TOPOLOGY_CANDIDATES[candidate_name]
                x_dict, edge_index_dict, metadata, node_input_dims = candidate_graph_inputs[candidate_name]
                model = DisruptionCalibratedHGT(
                    metadata=metadata,
                    node_input_dims=node_input_dims,
                    edge_feature_dim=edge_feature_tensors[candidate_name].shape[1],
                    disruption_dim=disruption_tensors[disruption_group].shape[1],
                    num_years=len(year_to_idx),
                    hidden_dim=cfg.hidden_dim,
                    num_layers=cfg.num_layers,
                    num_heads=cfg.num_heads,
                    dropout=cfg.dropout,
                    gate_alpha=cfg.disruption_gate_alpha,
                ).to(device)
                model.load_state_dict(best_states[metric_name])

                if cfg.save_models:
                    model_path = paths.models_dir / f"{candidate_name}__{disruption_group}__seed{seed}__{metric_name}.pt"
                    torch.save({"state_dict": best_states[metric_name], "config": asdict(cfg)}, model_path)

                val_pred = evaluate_model_on_indices(model, candidate_name, disruption_group, val_indices)
                test_pred = evaluate_model_on_indices(model, candidate_name, disruption_group, test_indices)
                pred = pd.concat([val_pred, test_pred], ignore_index=True)
                pred["source"] = "DCQHGT_DISRUPTION"
                pred["model"] = candidate_name
                pred["variant"] = disruption_group
                pred["checkpoint_metric"] = metric_name
                pred["seed"] = str(seed)
                prediction_tables.append(pred)

elapsed = time.time() - start_time
print(f"Training completed in {elapsed / 60:.2f} minutes.")

training_history = pd.DataFrame(all_history_rows)
checkpoint_summary = pd.DataFrame(checkpoint_rows)
graph_predictions_by_seed = pd.concat(prediction_tables, ignore_index=True) if prediction_tables else pd.DataFrame()

log_frame("Checkpoint summary", checkpoint_summary, max_rows=20)
log_frame("Predictions by seed", graph_predictions_by_seed, max_rows=5)



Training candidate=hgt_truck_rail_plus_topology_features, disruption_group=no_disruption, seed=7
[hgt_truck_rail_plus_topology_features | no_disruption | seed 7] epoch=1 train_loss=481.5992 val_wpin=363.038 val_q75=1170.895 val_iqr=776.714
[hgt_truck_rail_plus_topology_features | no_disruption | seed 7] epoch=20 train_loss=142.2328 val_wpin=147.332 val_q75=528.698 val_iqr=429.494
[hgt_truck_rail_plus_topology_features | no_disruption | seed 7] epoch=40 train_loss=90.9626 val_wpin=123.060 val_q75=444.588 val_iqr=355.616
[hgt_truck_rail_plus_topology_features | no_disruption | seed 7] epoch=60 train_loss=76.2750 val_wpin=127.864 val_q75=458.258 val_iqr=360.650
[hgt_truck_rail_plus_topology_features | no_disruption | seed 7] epoch=80 train_loss=68.3817 val_wpin=132.525 val_q75=458.490 val_iqr=360.574
[hgt_truck_rail_plus_topology_features | no_disruption | seed 7] epoch=100 train_loss=65.3287 val_wpin=129.534 val_q75=458.508 val_iqr=357.993
[hgt_truck_rail_plus_topology_features | no_dis


## 17. Build 5-seed ensemble predictions

For each topology candidate, disruption group, and checkpoint metric, this cell averages predictions across the five seeds. Seed ensembles are reported separately from single-seed metrics.


In [17]:

# ---------------------------------------------------------------------
# Build seed ensemble predictions.
# ---------------------------------------------------------------------

ensemble_group_cols = ["source", "model", "variant", "checkpoint_metric", "split", "row_id"]
value_cols = ["true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75", "sample_weight"]
carry_cols = ["year", "n_fmi_county_pairs", "obs_weight_sum", "faf_orig", "faf_dest"]

ensemble_rows = []
if not graph_predictions_by_seed.empty:
    for group_key, group in graph_predictions_by_seed.groupby(["model", "variant", "checkpoint_metric", "split", "row_id"], dropna=False):
        model, variant, checkpoint_metric, split, row_id = group_key
        row = {
            "source": "DCQHGT_DISRUPTION_ENSEMBLE",
            "model": model,
            "variant": variant,
            "checkpoint_metric": checkpoint_metric,
            "seed": "ensemble",
            "split": split,
            "row_id": int(row_id),
        }
        for col in ["true_q25", "true_q50", "true_q75", "sample_weight"]:
            row[col] = float(group[col].iloc[0])
        for col in ["pred_q25", "pred_q50", "pred_q75"]:
            row[col] = float(group[col].mean())
        for col in carry_cols:
            if col in group.columns:
                row[col] = group[col].iloc[0]
        ensemble_rows.append(row)

graph_predictions_seed_ensemble = pd.DataFrame(ensemble_rows)
log_frame("Seed ensemble predictions", graph_predictions_seed_ensemble, max_rows=5)



Seed ensemble predictions: shape=(108756, 19)
                    source                             model            variant checkpoint_metric     seed split  row_id    true_q25    true_q50    true_q75  sample_weight    pred_q25    pred_q50    pred_q75  year  n_fmi_county_pairs  obs_weight_sum faf_orig faf_dest
DCQHGT_DISRUPTION_ENSEMBLE hgt_terminal_plus_truck_plus_rail border_global_only      best_val_iqr ensemble  test   63410  956.929993 1384.719971 2500.750000       1.173914 1178.813477 1566.721924 2614.733643  2024                  77      681.565142      011      123
DCQHGT_DISRUPTION_ENSEMBLE hgt_terminal_plus_truck_plus_rail border_global_only      best_val_iqr ensemble  test   63428  416.829987 1030.079956 1532.829956       1.291330  904.465698 1343.018188 2192.247559  2024                 106     1310.047008      011      212
DCQHGT_DISRUPTION_ENSEMBLE hgt_terminal_plus_truck_plus_rail border_global_only      best_val_iqr ensemble  test   63441 1456.869995 2388.030029 3769


## 18. Load previous baselines and build combined prediction table

This cell loads previous Cold-OD and graph/topology predictions when available. They are used for combined leaderboard and paired comparisons.


In [18]:

# ---------------------------------------------------------------------
# Load previous baselines and build combined predictions.
# ---------------------------------------------------------------------

def normalize_prediction_schema(frame: pd.DataFrame, default_source: str) -> pd.DataFrame:
    """Normalize baseline prediction tables to the current schema."""
    if frame.empty:
        return frame
    out = frame.copy()
    if "source" not in out.columns:
        out["source"] = default_source
    if "variant" not in out.columns:
        out["variant"] = "baseline"
    if "checkpoint_metric" not in out.columns:
        out["checkpoint_metric"] = "baseline"
    if "seed" not in out.columns:
        out["seed"] = "baseline"
    if "row_id" not in out.columns:
        # Fallback to index; this should be avoided for paired comparisons.
        out["row_id"] = np.arange(len(out))
    rename_map = {
        "actual_q25": "true_q25", "actual_q50": "true_q50", "actual_q75": "true_q75",
        "y_q25": "true_q25", "y_q50": "true_q50", "y_q75": "true_q75",
        "q25_true": "true_q25", "q50_true": "true_q50", "q75_true": "true_q75",
        "q25_pred": "pred_q25", "q50_pred": "pred_q50", "q75_pred": "pred_q75",
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})
    needed = ["source", "model", "variant", "checkpoint_metric", "seed", "split", "row_id", "true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75"]
    keep = [col for col in needed + ["sample_weight", "n_fmi_county_pairs", "obs_weight_sum", "faf_orig", "faf_dest", "year"] if col in out.columns]
    return out.loc[:, keep].copy()

baseline_tables = []
for path, source in [
    (paths.cold_baseline_predictions_path, "ColdOD"),
    (paths.graphv2_predictions_path, "GraphV2"),
    (paths.topology_v3_predictions_path, "TopologyV3"),
]:
    if Path(path).exists():
        try:
            baseline = pd.read_parquet(path)
            baseline = normalize_prediction_schema(baseline, source)
            baseline_tables.append(baseline)
            print(f"Loaded baseline predictions: {source}, shape={baseline.shape}")
        except Exception as exc:
            print(f"Warning: failed to load {source} predictions from {path}: {exc}")

combined_predictions = pd.concat(
    baseline_tables + [graph_predictions_by_seed, graph_predictions_seed_ensemble],
    ignore_index=True,
    sort=False,
) if baseline_tables else pd.concat([graph_predictions_by_seed, graph_predictions_seed_ensemble], ignore_index=True, sort=False)

log_frame("Combined predictions", combined_predictions, max_rows=5)


Loaded baseline predictions: ColdOD, shape=(22154, 19)
Loaded baseline predictions: GraphV2, shape=(94658, 18)
Loaded baseline predictions: TopologyV3, shape=(515584, 18)

Combined predictions: shape=(1284932, 19)
source               model       variant checkpoint_metric     seed split  row_id  true_q25  true_q50  true_q75  pred_q25  pred_q50  pred_q75  sample_weight  n_fmi_county_pairs  obs_weight_sum faf_orig faf_dest  year
ColdOD global_train_median postprocessed          baseline baseline   val       0   1821.78   2695.00   4182.33   1493.07   2317.03   3636.85       0.249376                 3.0        3.000000      011      109  2023
ColdOD global_train_median postprocessed          baseline baseline   val       1   1325.63   1769.42   3000.07   1493.07   2317.03   3636.85       0.249376                 3.0        3.000000      011      111  2023
ColdOD global_train_median postprocessed          baseline baseline   val       2   1480.87   2529.27   4608.47   1493.07   2317.03   3


## 19. Metrics, leaderboard, and seed summaries

This cell computes metrics for single-seed predictions, seed ensembles, and combined baselines. The main leaderboard uses `split == 'test'` only.


In [19]:

# ---------------------------------------------------------------------
# Metrics, leaderboard, and seed summaries.
# ---------------------------------------------------------------------

def compute_metrics_table(predictions: pd.DataFrame) -> pd.DataFrame:
    """Compute metrics for each prediction group."""
    rows = []
    if predictions.empty:
        return pd.DataFrame()
    group_cols = ["source", "model", "variant", "checkpoint_metric", "seed", "split"]
    for key, group in predictions.groupby(group_cols, dropna=False):
        metrics = compute_metrics_from_arrays(group)
        row = dict(zip(group_cols, key))
        row.update(metrics)
        rows.append(row)
    return pd.DataFrame(rows)

metrics_by_seed = compute_metrics_table(graph_predictions_by_seed)
metrics_seed_ensemble = compute_metrics_table(graph_predictions_seed_ensemble)
metrics_combined = compute_metrics_table(combined_predictions)

leaderboard_test = (
    metrics_combined.query("split == 'test'")
    .sort_values("pinball_mean", ascending=True)
    .reset_index(drop=True)
)
leaderboard_test.insert(0, "rank", np.arange(1, len(leaderboard_test) + 1))

seed_summary = (
    metrics_by_seed.query("split == 'test'")
    .groupby(["source", "model", "variant", "checkpoint_metric"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        pinball_mean_mean=("pinball_mean", "mean"),
        pinball_mean_std=("pinball_mean", "std"),
        weighted_pinball_mean_mean=("weighted_pinball_mean", "mean"),
        mae_q75_mean=("mae_q75", "mean"),
        mae_q75_std=("mae_q75", "std"),
        iqr_mae_mean=("iqr_mae", "mean"),
        iqr_mae_std=("iqr_mae", "std"),
        stress_top10_mae_q75_mean=("stress_top10_mae_q75", "mean"),
        sparse_bottom25_mae_q75_mean=("sparse_bottom25_mae_q75", "mean"),
    )
    .sort_values("pinball_mean_mean")
)

log_frame("Test leaderboard", leaderboard_test, max_rows=20)
log_frame("Seed summary", seed_summary, max_rows=20)



Test leaderboard: shape=(482, 32)
 rank                      source                                 model       variant checkpoint_metric     seed split  n_rows    mae_q25    mae_q50    mae_q75  pinball_q25  pinball_q50  pinball_q75  pinball_mean    iqr_mae    bias_q25   bias_q50   bias_q75  raw_crossing_rate  weighted_pinball_mean  stress_top10_threshold_q75  stress_top10_n  stress_top10_mae_q75  top_iqr10_threshold  top_iqr10_n  top_iqr10_mae_q75  top_iqr10_iqr_mae  sparse_bottom25_threshold_n_fmi  sparse_bottom25_n  sparse_bottom25_mae_q75  sparse_bottom25_iqr_mae
    1              GRAPH_ENSEMBLE           hgt_residual_prior_features postprocessed      best_val_iqr     mean  test    1057 163.551925 223.604117 453.218931    67.491518   111.802059   207.196589    128.830055 377.083895  -57.137779 -14.037282  77.651505                0.0             117.697322                 6192.161914             106            519.854465          3613.559961          106         518.193018       


## 20. Strict test-only segment summaries

This cell builds test-only segment summaries for true q75 deciles, true IQR deciles, and sparse-label quartiles. These outputs are used to evaluate stress and sparse Cold-OD behavior.


In [20]:

# ---------------------------------------------------------------------
# Strict test-only segment summaries.
# ---------------------------------------------------------------------

def add_segment_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Add true q75 / IQR deciles and n_fmi quartiles to a prediction frame."""
    out = frame.copy()
    test_mask = out["split"].astype(str).eq("test")
    test = out.loc[test_mask]
    if test.empty:
        return out

    q75_bins = pd.qcut(test["true_q75"], q=10, duplicates="drop")
    iqr_values = test["true_q75"] - test["true_q25"]
    iqr_bins = pd.qcut(iqr_values, q=10, duplicates="drop")
    out.loc[test.index, "true_q75_decile"] = [f"true_q75_decile_{i+1:02d}" for i in q75_bins.cat.codes]
    out.loc[test.index, "true_iqr_decile"] = [f"true_iqr_decile_{i+1:02d}" for i in iqr_bins.cat.codes]

    if "n_fmi_county_pairs" in test.columns:
        n_fmi_bins = pd.qcut(test["n_fmi_county_pairs"], q=4, duplicates="drop")
        out.loc[test.index, "n_fmi_county_pairs_quartile"] = [f"n_fmi_q{i+1}" for i in n_fmi_bins.cat.codes]
    return out


def segment_summary(frame: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    """Compute metrics by model and segment for test rows only."""
    test = frame.query("split == 'test'").copy()
    if test.empty or segment_col not in test.columns:
        return pd.DataFrame()
    rows = []
    group_cols = ["source", "model", "variant", "checkpoint_metric", "seed", segment_col]
    for key, group in test.groupby(group_cols, dropna=False):
        metrics = compute_metrics_from_arrays(group)
        row = dict(zip(group_cols, key))
        row.update(metrics)
        rows.append(row)
    return pd.DataFrame(rows)

combined_predictions = add_segment_columns(combined_predictions)
segment_tables = {
    "true_q75_decile": segment_summary(combined_predictions, "true_q75_decile"),
    "true_iqr_decile": segment_summary(combined_predictions, "true_iqr_decile"),
    "n_fmi_county_pairs_quartile": segment_summary(combined_predictions, "n_fmi_county_pairs_quartile"),
}

for name, table in segment_tables.items():
    print(name, table.shape)
    if not table.empty:
        log_frame(name, table.head(10))


true_q75_decile (4820, 31)

true_q75_decile: shape=(10, 31)
source             model       variant checkpoint_metric     seed    true_q75_decile  n_rows     mae_q25     mae_q50     mae_q75  pinball_q25  pinball_q50  pinball_q75  pinball_mean     iqr_mae    bias_q25    bias_q50    bias_q75  raw_crossing_rate  weighted_pinball_mean  stress_top10_threshold_q75  stress_top10_n  stress_top10_mae_q75  top_iqr10_threshold  top_iqr10_n  top_iqr10_mae_q75  top_iqr10_iqr_mae  sparse_bottom25_threshold_n_fmi  sparse_bottom25_n  sparse_bottom25_mae_q75  sparse_bottom25_iqr_mae
ColdOD destination_prior postprocessed          baseline baseline true_q75_decile_01     105 1331.660762 1840.735143 2586.235571   998.745571   920.367571   646.558893    855.224012 1254.574810 1331.660762 1840.735143 2586.235571                0.0             852.637339                    1424.616              11           2157.543636             1188.540           11        1773.956364         579.277273                   


## 21. Robust paired summary

This cell creates strict test-only paired comparisons. It automatically selects available references and pairs by `row_id`.

The most important references are:

- ColdOD MLP residual prior,
- GraphV2 HGT baseline if available,
- GraphV2 GraphSAGE baseline if available,
- topology-v3 no-topology HGT if available,
- internal no-disruption variants from this notebook.


In [21]:

# ---------------------------------------------------------------------
# Robust strict test-only paired summary.
# ---------------------------------------------------------------------

def select_reference_rows(predictions: pd.DataFrame) -> pd.DataFrame:
    """Select robust reference prediction groups for paired comparison."""
    test = predictions.query("split == 'test'").copy()
    refs = []
    candidates = [
        ("coldod_mlp_residual_prior", lambda df: df["source"].astype(str).str.contains("ColdOD") & df["model"].astype(str).str.contains("mlp_residual_prior_features")),
        ("graphv2_hgt", lambda df: df["source"].astype(str).str.contains("GraphV2") & df["model"].astype(str).str.contains("hgt_residual_prior_features")),
        ("graphv2_graphsage", lambda df: df["source"].astype(str).str.contains("GraphV2") & df["model"].astype(str).str.contains("graphsage_residual_prior_features")),
        ("topology_v3_no_topology", lambda df: df["source"].astype(str).str.contains("TopologyV3") & df["model"].astype(str).str.contains("faf_hgt_no_topology")),
        ("current_no_disruption_truck_rail", lambda df: df["source"].astype(str).str.contains("DCQHGT_DISRUPTION") & df["model"].astype(str).str.contains("hgt_truck_rail_plus_topology_features") & df["variant"].astype(str).eq("no_disruption")),
        ("current_no_disruption_risk", lambda df: df["source"].astype(str).str.contains("DCQHGT_DISRUPTION") & df["model"].astype(str).str.contains("hgt_terminal_plus_truck_plus_rail") & df["variant"].astype(str).eq("no_disruption")),
    ]
    for name, selector in candidates:
        subset = test.loc[selector(test)].copy()
        if subset.empty:
            continue
        # Prefer ensemble rows, then best_val_pinball, then first available group.
        subset["_prefer_ensemble"] = subset["seed"].astype(str).eq("ensemble").astype(int)
        subset["_prefer_pinball"] = subset["checkpoint_metric"].astype(str).eq("best_val_pinball").astype(int)
        group_cols = ["source", "model", "variant", "checkpoint_metric", "seed"]
        group_sizes = subset.groupby(group_cols).size().reset_index(name="n_rows")
        group_sizes["_prefer_ensemble"] = group_sizes["seed"].astype(str).eq("ensemble").astype(int)
        group_sizes["_prefer_pinball"] = group_sizes["checkpoint_metric"].astype(str).eq("best_val_pinball").astype(int)
        chosen = group_sizes.sort_values(["_prefer_ensemble", "_prefer_pinball", "n_rows"], ascending=[False, False, False]).iloc[0]
        mask = np.ones(len(subset), dtype=bool)
        for col in group_cols:
            mask &= subset[col].astype(str).eq(str(chosen[col])).to_numpy()
        chosen_rows = subset.loc[mask].copy()
        chosen_rows["reference_name"] = name
        refs.append(chosen_rows.drop(columns=["_prefer_ensemble", "_prefer_pinball"], errors="ignore"))
    return pd.concat(refs, ignore_index=True) if refs else pd.DataFrame()


def build_paired_summary(predictions: pd.DataFrame, references: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Build strict test-only paired rows and summary metrics by row_id."""
    if references.empty:
        return pd.DataFrame(), pd.DataFrame()

    test = predictions.query("split == 'test'").copy()
    candidate_mask = test["source"].astype(str).str.contains("DCQHGT_DISRUPTION")
    candidates = test.loc[candidate_mask].copy()

    pair_rows = []
    summary_rows = []
    ref_group_cols = ["reference_name", "source", "model", "variant", "checkpoint_metric", "seed"]
    cand_group_cols = ["source", "model", "variant", "checkpoint_metric", "seed"]

    for ref_key, ref_group in references.groupby(ref_group_cols, dropna=False):
        ref_name = ref_key[0]
        ref_cols = ref_group[["row_id", "pred_q25", "pred_q50", "pred_q75", "true_q25", "true_q50", "true_q75"]].copy()
        ref_cols = ref_cols.rename(columns={
            "pred_q25": "ref_pred_q25", "pred_q50": "ref_pred_q50", "pred_q75": "ref_pred_q75",
        })

        for cand_key, cand_group in candidates.groupby(cand_group_cols, dropna=False):
            merged = cand_group.merge(ref_cols, on=["row_id", "true_q25", "true_q50", "true_q75"], how="inner")
            if merged.empty:
                continue
            true = merged[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=np.float64)
            pred = merged[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=np.float64)
            ref = merged[["ref_pred_q25", "ref_pred_q50", "ref_pred_q75"]].to_numpy(dtype=np.float64)

            diff_c = true - pred
            diff_r = true - ref
            pin_c = np.maximum(TAUS_CPU.reshape(1, 3) * diff_c, (TAUS_CPU.reshape(1, 3) - 1.0) * diff_c).mean(axis=1)
            pin_r = np.maximum(TAUS_CPU.reshape(1, 3) * diff_r, (TAUS_CPU.reshape(1, 3) - 1.0) * diff_r).mean(axis=1)
            q75_c = np.abs(pred[:, 2] - true[:, 2])
            q75_r = np.abs(ref[:, 2] - true[:, 2])
            iqr_c = np.abs((pred[:, 2] - pred[:, 0]) - (true[:, 2] - true[:, 0]))
            iqr_r = np.abs((ref[:, 2] - ref[:, 0]) - (true[:, 2] - true[:, 0]))

            merged["reference_name"] = ref_name
            merged["candidate_source"] = cand_key[0]
            merged["candidate_model"] = cand_key[1]
            merged["candidate_variant"] = cand_key[2]
            merged["candidate_checkpoint_metric"] = cand_key[3]
            merged["candidate_seed"] = cand_key[4]
            merged["delta_pinball"] = pin_r - pin_c
            merged["delta_abs_error_q75"] = q75_r - q75_c
            merged["delta_iqr_abs_error"] = iqr_r - iqr_c
            pair_rows.append(merged)

            summary_rows.append({
                "reference_name": ref_name,
                "candidate_source": cand_key[0],
                "candidate_model": cand_key[1],
                "candidate_variant": cand_key[2],
                "candidate_checkpoint_metric": cand_key[3],
                "candidate_seed": cand_key[4],
                "n_rows": int(len(merged)),
                "mean_delta_pinball": float(np.mean(pin_r - pin_c)),
                "win_rate_pinball": float(np.mean((pin_r - pin_c) > 0)),
                "mean_delta_abs_error_q75": float(np.mean(q75_r - q75_c)),
                "win_rate_abs_error_q75": float(np.mean((q75_r - q75_c) > 0)),
                "mean_delta_iqr_abs_error": float(np.mean(iqr_r - iqr_c)),
                "win_rate_iqr_abs_error": float(np.mean((iqr_r - iqr_c) > 0)),
            })

    return pd.DataFrame(summary_rows), pd.concat(pair_rows, ignore_index=True) if pair_rows else pd.DataFrame()

reference_rows = select_reference_rows(combined_predictions)
paired_summary, paired_rows = build_paired_summary(combined_predictions, reference_rows)

reference_selection = reference_rows.groupby(["reference_name", "source", "model", "variant", "checkpoint_metric", "seed"], as_index=False).size().rename(columns={"size": "n_rows"}) if not reference_rows.empty else pd.DataFrame()

log_frame("Paired reference selection", reference_selection, max_rows=20)
log_frame("Paired summary", paired_summary.sort_values("mean_delta_pinball", ascending=False) if not paired_summary.empty else paired_summary, max_rows=20)



Paired reference selection: shape=(5, 7)
                  reference_name                     source                                 model       variant checkpoint_metric     seed  n_rows
       coldod_mlp_residual_prior                     ColdOD           mlp_residual_prior_features postprocessed          baseline baseline    1057
      current_no_disruption_risk DCQHGT_DISRUPTION_ENSEMBLE     hgt_terminal_plus_truck_plus_rail no_disruption  best_val_pinball ensemble    1057
current_no_disruption_truck_rail DCQHGT_DISRUPTION_ENSEMBLE hgt_truck_rail_plus_topology_features no_disruption  best_val_pinball ensemble    1057
               graphv2_graphsage             GraphV2::GRAPH     graphsage_residual_prior_features      baseline          baseline      123    3171
                     graphv2_hgt             GraphV2::GRAPH           hgt_residual_prior_features      baseline          baseline      123    3171

Paired summary: shape=(972, 13)
reference_name  candidate_source           


## 22. Optional plots

This cell creates two lightweight diagnostic plots if matplotlib is available.


In [22]:

# ---------------------------------------------------------------------
# Optional diagnostic plots.
# ---------------------------------------------------------------------

if cfg.make_plots and MATPLOTLIB_AVAILABLE and not leaderboard_test.empty:
    plot_df = leaderboard_test.head(25).iloc[::-1].copy()
    labels = plot_df["source"].astype(str) + "::" + plot_df["model"].astype(str) + "::" + plot_df["variant"].astype(str) + "::" + plot_df["checkpoint_metric"].astype(str)
    plt.figure(figsize=(12, 10))
    plt.barh(labels, plot_df["pinball_mean"])
    plt.xlabel("pinball_mean")
    plt.title("Cold-OD D-CQHGT Disruption Test Pinball Mean")
    plt.tight_layout()
    plt.savefig(paths.plots_dir / "dcqhgt_disruption_leaderboard_pinball_mean.png", dpi=160)
    plt.close()

if cfg.make_plots and MATPLOTLIB_AVAILABLE and "true_q75_decile" in segment_tables and not segment_tables["true_q75_decile"].empty:
    seg = segment_tables["true_q75_decile"].copy()
    # Plot only a manageable subset: no-disruption and full disruption ensembles.
    mask = (
        seg["source"].astype(str).str.contains("DCQHGT_DISRUPTION_ENSEMBLE")
        & seg["variant"].astype(str).isin(["no_disruption", "weather_zone_only", "full_disruption_variable_only"])
        & seg["checkpoint_metric"].astype(str).eq("best_val_pinball")
    )
    plot_seg = seg.loc[mask].copy()
    if not plot_seg.empty:
        plt.figure(figsize=(16, 8))
        for key, group in plot_seg.groupby(["model", "variant"]):
            group = group.sort_values("true_q75_decile")
            plt.plot(group["true_q75_decile"], group["mae_q75"], marker="o", label="::".join(key))
        plt.xticks(rotation=45, ha="right")
        plt.ylabel("mae_q75")
        plt.xlabel("true_q75_decile")
        plt.title("Cold-OD q75 MAE by True q75 Decile: Disruption Ablation")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(paths.plots_dir / "dcqhgt_disruption_q75_mae_by_true_q75_decile.png", dpi=160)
        plt.close()



## 23. Save output artifacts

This cell saves all predictions, metrics, feature manifests, paired summaries, segment summaries, configuration files, and a compact plain-text report. It uses safe Parquet writing and does not require optional packages such as `tabulate`.


In [23]:

# ---------------------------------------------------------------------
# Save output artifacts.
# ---------------------------------------------------------------------

# Prediction outputs.
by_seed_path = paths.output_dir / "predictions_dcqhgt_disruption_val_test_by_seed.parquet"
ensemble_path = paths.output_dir / "predictions_dcqhgt_disruption_val_test_seed_ensemble.parquet"
combined_path = paths.output_dir / "combined_predictions_dcqhgt_disruption_val_test.parquet"

safe_to_parquet(graph_predictions_by_seed, by_seed_path)
safe_to_parquet(graph_predictions_seed_ensemble, ensemble_path)
safe_to_parquet(combined_predictions, combined_path)

# Metrics outputs.
metrics_by_seed_path = paths.output_dir / "metrics_dcqhgt_disruption_by_seed.csv"
metrics_ensemble_path = paths.output_dir / "metrics_dcqhgt_disruption_seed_ensemble.csv"
metrics_combined_path = paths.output_dir / "metrics_dcqhgt_disruption_combined.csv"
leaderboard_path = paths.output_dir / "leaderboard_test_dcqhgt_disruption.csv"
seed_summary_path = paths.output_dir / "seed_summary_dcqhgt_disruption.csv"
checkpoint_summary_path = paths.output_dir / "checkpoint_summary_dcqhgt_disruption.csv"
history_path = paths.output_dir / "training_history_dcqhgt_disruption.csv"

metrics_by_seed.to_csv(metrics_by_seed_path, index=False)
metrics_seed_ensemble.to_csv(metrics_ensemble_path, index=False)
metrics_combined.to_csv(metrics_combined_path, index=False)
leaderboard_test.to_csv(leaderboard_path, index=False)
seed_summary.to_csv(seed_summary_path, index=False)
checkpoint_summary.to_csv(checkpoint_summary_path, index=False)
training_history.to_csv(history_path, index=False)

# Feature pruning and group manifests.
feature_pruning_audit.to_csv(paths.tables_dir / "disruption_feature_pruning_audit.csv", index=False)
write_json(disruption_groups, paths.output_dir / "disruption_group_manifest.json")

# Segment summaries.
for name, table in segment_tables.items():
    table.to_csv(paths.tables_dir / f"dcqhgt_disruption_test_only_segment_summary__{name}.csv", index=False)

# Paired summaries.
paired_summary_path = paths.output_dir / "paired_summary_dcqhgt_disruption_test_only.csv"
paired_rows_path = paths.output_dir / "paired_rows_dcqhgt_disruption_test_only.parquet"
reference_selection_path = paths.tables_dir / "dcqhgt_disruption_paired_reference_selection.csv"
paired_summary.to_csv(paired_summary_path, index=False)
reference_selection.to_csv(reference_selection_path, index=False)
if not paired_rows.empty:
    safe_to_parquet(paired_rows, paired_rows_path)

# Preprocessor metadata.
preprocessor_payload = {
    "edge_preprocessors": {name: prep.to_dict() for name, prep in edge_preprocessors.items()},
    "disruption_preprocessors": {name: prep.to_dict() for name, prep in disruption_preprocessors.items()},
    "target_scale": target_scale,
    "year_to_idx": {str(k): int(v) for k, v in year_to_idx.items()},
}
write_json(preprocessor_payload, paths.output_dir / "preprocessors_dcqhgt_disruption.json")

run_config = {
    "notebook": "DCQHGT_Disruption_ColdOD",
    "config": asdict(cfg),
    "paths": {
        "heterodata_path": str(paths.heterodata_path),
        "supervised_path": str(paths.supervised_path),
        "supervised_with_disruption_path": str(paths.supervised_with_disruption_path),
        "disruption_qa_summary_path": str(paths.disruption_qa_summary_path),
        "output_dir": str(paths.output_dir),
    },
    "dataset": {
        "n_rows": int(len(supervised_disruption_df)),
        "cold_train_rows": int(cold_train_mask.sum()),
        "cold_val_rows": int(cold_val_mask.sum()),
        "cold_test_rows": int(cold_test_mask.sum()),
        "manifest_feature_count": len(manifest_feature_columns),
        "edge_attr_dim": int(edge_attr_raw_all.shape[1]),
    },
    "package_versions": {
        "python": os.sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
    },
}
write_json(run_config, paths.output_dir / "run_config_dcqhgt_disruption.json")

artifact_manifest = {
    "predictions_by_seed": str(by_seed_path),
    "predictions_seed_ensemble": str(ensemble_path),
    "combined_predictions": str(combined_path),
    "leaderboard_test": str(leaderboard_path),
    "metrics_by_seed": str(metrics_by_seed_path),
    "metrics_seed_ensemble": str(metrics_ensemble_path),
    "metrics_combined": str(metrics_combined_path),
    "seed_summary": str(seed_summary_path),
    "checkpoint_summary": str(checkpoint_summary_path),
    "training_history": str(history_path),
    "paired_summary": str(paired_summary_path),
    "paired_rows": str(paired_rows_path) if not paired_rows.empty else None,
    "tables_dir": str(paths.tables_dir),
    "plots_dir": str(paths.plots_dir),
    "models_dir": str(paths.models_dir),
}
write_json(artifact_manifest, paths.output_dir / "analysis_artifact_manifest_dcqhgt_disruption.json")

# Plain-text report.
report_lines = []
report_lines.append("# D-CQHGT Disruption Cold-OD Report")
report_lines.append("")
report_lines.append(f"Output directory: {paths.output_dir}")
report_lines.append(f"Topology candidates: {list(cfg.enabled_topology_candidates)}")
report_lines.append(f"Disruption groups: {list(disruption_groups.keys())}")
report_lines.append(f"Seeds: {list(cfg.seeds)}")
report_lines.append("")
report_lines.append("## Top test leaderboard rows")
report_lines.append(dataframe_to_text(leaderboard_test, ["rank", "source", "model", "variant", "checkpoint_metric", "seed", "pinball_mean", "weighted_pinball_mean", "mae_q75", "iqr_mae", "stress_top10_mae_q75"], max_rows=25))
report_lines.append("")
report_lines.append("## Seed summary")
report_lines.append(dataframe_to_text(seed_summary, None, max_rows=25))
report_lines.append("")
report_lines.append("## Paired summary")
report_lines.append(dataframe_to_text(paired_summary.sort_values("mean_delta_pinball", ascending=False) if not paired_summary.empty else paired_summary, None, max_rows=25))

report_path = paths.reports_dir / "dcqhgt_disruption_cold_od_report.txt"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved D-CQHGT disruption artifacts to:", paths.output_dir)
print(json.dumps(artifact_manifest, indent=2))


Saved D-CQHGT disruption artifacts to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\dcqhgt_disruption_cold_od_v1_notebook\east_plus_gulf
{
  "predictions_by_seed": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\dcqhgt_disruption_cold_od_v1_notebook\\east_plus_gulf\\predictions_dcqhgt_disruption_val_test_by_seed.parquet",
  "predictions_seed_ensemble": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\dcqhgt_disruption_cold_od_v1_notebook\\east_plus_gulf\\predictions_dcqhgt_disruption_val_test_seed_ensemble.parquet",
  "combined_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\dcqhgt_disruption_cold_od_v1_notebook\\east_plus_gulf\\combined_predictions_dcqhgt_disruption_val_test.parquet",
  "leaderboard_test": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\dcqhgt_disruption_cold_od_v1_notebook\\east_plus_gulf\\leaderboard_test_dcqhgt_disruption.csv",
  "metrics_by_seed": "E:\\NetworkOptimization\\py